In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from src.data_loader import load_goemotions, load_hf_emotion

go_df = load_goemotions()
hf_df = load_hf_emotion()

d:\Portfolio\Machine_Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GoEmotions from local...
Loading HF Emotion from local...


In [3]:
print(hf_df['label'].unique())

[0 3 2 5 4 1]


In [4]:
from datasets import load_dataset

hf_dataset = load_dataset("emotion")

hf_label_names = hf_dataset["train"].features["label"].names

print(hf_label_names)

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [5]:
HF_LABEL_NAMES = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

print(HF_LABEL_NAMES)

{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [6]:
print(go_df['labels'].head())
print(type(go_df['labels'][0]))

0    [27]
1    [27]
2     [2]
3    [14]
4     [3]
Name: labels, dtype: str
<class 'str'>


In [7]:
from datasets import load_dataset

go_dataset = load_dataset("go_emotions")

go_label_names = go_dataset["train"].features["labels"].feature.names

print(go_label_names)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [8]:
print(go_label_names[27])

neutral


In [9]:
import ast

go_df['labels'] = go_df['labels'].apply(lambda x: [int(i) for i in ast.literal_eval(x)])

In [10]:
sample = go_df.iloc[0]

label_ids = sample['labels']

mapped = [go_label_names[int(i)] for i in label_ids]

print("Original:", label_ids)
print("Mapped:", mapped)

Original: [27]
Mapped: ['neutral']


In [11]:
PRIMARY_EMOTIONS = [
    "happy",
    "sad",
    "angry",
    "fear",
    "surprise",
    "neutral"
]

In [12]:
GOEMOTION_MAP = {
    "admiration": ("happy", "proud"),
    "amusement": ("happy", "joyful"),
    "anger": ("angry", "angry"),
    "annoyance": ("angry", "annoyed"),
    "disappointment": ("sad", "disappointed"),
    "grief": ("sad", "hopeless"),
    "fear": ("fear", "fear"),
    "nervousness": ("fear", "nervous"),
    "joy": ("happy", "joyful"),
    "love": ("happy", "love"),
    "optimism": ("happy", "excited"),
    "sadness": ("sad", "sad"),
    "surprise": ("surprise", "surprised"),
    "neutral": ("neutral", "neutral")
}

In [13]:
HF_MAP = {
    "sadness": ("sad", "sad"),
    "joy": ("happy", "joyful"),
    "love": ("happy", "love"),
    "anger": ("angry", "angry"),
    "fear": ("fear", "fear"),
    "surprise": ("surprise", "surprised")
}

In [14]:
hf_dataset = load_dataset("emotion")

hf_label_names = hf_dataset["train"].features["label"].names

sample = hf_df.iloc[0]

emotion = hf_label_names[sample['label']]

mapped = HF_MAP[emotion]

print("Original:", emotion)
print("Mapped:", mapped)

Original: sadness
Mapped: ('sad', 'sad')


In [15]:
sample = go_df.iloc[0]

label_ids = sample['labels']

emotions = [go_label_names[i] for i in label_ids]

mapped = [GOEMOTION_MAP[e] for e in emotions if e in GOEMOTION_MAP]

print("Original:", emotions)
print("Mapped:", mapped)

Original: ['neutral']
Mapped: [('neutral', 'neutral')]


In [16]:
from src.preprocess import process_hf, process_goemotions
import pandas as pd
from datasets import load_dataset

# Load label names
hf_dataset = load_dataset("emotion")
hf_label_names = hf_dataset["train"].features["label"].names

go_dataset = load_dataset("go_emotions")
go_label_names = go_dataset["train"].features["labels"].feature.names

# Process datasets
hf_processed = process_hf(hf_df, hf_label_names)
go_processed = process_goemotions(go_df, go_label_names)

# Combine
final_df = pd.concat([hf_processed, go_processed], ignore_index=True)

print(final_df.head())

                                                text primary_emotion  \
0                            i didnt feel humiliated             sad   
1  i can go from feeling so hopeless to so damned...             sad   
2   im grabbing a minute to post i feel greedy wrong           angry   
3  i am ever feeling nostalgic about the fireplac...           happy   
4                               i am feeling grouchy           angry   

  sub_emotions  
0        [sad]  
1        [sad]  
2      [angry]  
3       [love]  
4      [angry]  


In [17]:
multi_label_samples = go_df[go_df['labels'].apply(lambda x: len(x) > 1)]

print(len(multi_label_samples))

7102


In [18]:
print(multi_label_samples.head())

                                                 text      labels       id
7   We need more boards and to create a bit more s...     [8, 20]  ef4qmod
11  Aww... she'll probably come around eventually,...      [1, 4]  edex4ki
15  Shit, I guess I accidentally bought a Pay-Per-...     [3, 12]  edivtm3
19  Maybe that’s what happened to the great white ...     [6, 22]  eczq8zg
20  I never thought it was at the same moment, but...  [6, 9, 27]  efdlhs1


In [19]:
sample = multi_label_samples.iloc[0]

labels = sample['labels']
emotions = [go_label_names[i] for i in labels]

mapped = [GOEMOTION_MAP[e][1] for e in emotions if e in GOEMOTION_MAP]

print(mapped)

['excited']


In [20]:
from src.data_loader import BASE_DIR

DATA_PATH = os.path.join(BASE_DIR, "data", "preprocessed")

os.makedirs(DATA_PATH, exist_ok=True)

In [21]:
final_df.to_csv(os.path.join(DATA_PATH, "final_dataset.csv"), index=False)